In [1]:
#I'm writing comments for the rabbit Leap problem for future reference
import itertools

# --- Problem Definition: Bridge Crossing Puzzle ---
# Four people need to cross a bridge at night with one umbrella.
# The goal is to get everyone across in 60 minutes or less.

 
PEOPLE = {
    'Amogh': 5,
    'Ameya': 10,
    'Grandmother': 20,
    'Grandfather': 25
}
TIME_LIMIT = 60

class State:

    # The goal is an empty tuple, representing no one left on the start side.
    GOAL_PEOPLE = ()

    def __init__(self, people_on_start, umbrella_at_start, time_elapsed=0):
         
        # We store people as a sorted tuple. This makes the state
        # immutable and hashable, and sorting ensures that ('A','B') is the
        # same as ('B','A'), which is crucial for the 'visited' check.
        self.people_on_start = tuple(sorted(people_on_start))
        self.umbrella_at_start = umbrella_at_start
        self.time_elapsed = time_elapsed
        
    def goalTest(self):
        #Checks if the current state is the goal state.
        return self.people_on_start == self.GOAL_PEOPLE

    def moveGen(self):
        #Generates all possible next states (children) from the current state.
        children = []
        
        # Convert the current tuple of people to a set for easier manipulation.
        current_people_set = set(self.people_on_start)
        
        # Case 1: Umbrella at start -> People cross to the end.
        if self.umbrella_at_start:
            for group_size in [1, 2]:
                for group in itertools.combinations(self.people_on_start, group_size):
                    trip_time = max(PEOPLE[person] for person in group)
                    new_time = self.time_elapsed + trip_time
                    
                    if new_time <= TIME_LIMIT:
                        new_people_set = current_people_set.difference(group)
                        children.append(State(new_people_set, False, new_time))
                        
        # Case 2: Umbrella at end -> One person returns to the start.
        else:
            all_people_set = set(PEOPLE.keys())
            people_on_end = all_people_set.difference(current_people_set)
            
            for person in people_on_end:
                trip_time = PEOPLE[person]
                new_time = self.time_elapsed + trip_time
                
                if new_time <= TIME_LIMIT:
                    new_people_set = current_people_set.union({person})
                    children.append(State(new_people_set, True, new_time))
                    
        return children

    # --- Helper methods for printing and comparison ---
    def __str__(self):
        """Returns a user-friendly string representation of the state."""
        start_side = list(self.people_on_start) or ["(None)"]
        end_side = sorted(list(set(PEOPLE.keys()).difference(self.people_on_start))) or ["(None)"]
        umbrella_pos = "START" if self.umbrella_at_start else "END"
        return f"Start Side: {start_side}, End Side: {end_side}, Umbrella at: {umbrella_pos}"

    def __repr__(self):
        """Returns a developer-friendly string representation for debugging."""
        return f"State({self.people_on_start}, {self.umbrella_at_start}, {self.time_elapsed})"

    def __eq__(self, other):
        """Checks for configurational equality between two states."""
        return (isinstance(other, State) and
                self.people_on_start == other.people_on_start and
                self.umbrella_at_start == other.umbrella_at_start)

    def __hash__(self):
        """Makes the State object hashable based on its configuration."""
        return hash((self.people_on_start, self.umbrella_at_start))


# --- Auxiliary Functions for Search Algorithms ---
def reconstructPath(node_pair, closed_list):
    """Traces back from the goal node to build the solution path."""
    path = []
    current_node, parent_node = node_pair
    path.append(current_node)
    
    parent_map = {node: parent for node, parent in closed_list}
    parent_map[current_node] = parent_node

    while parent_node is not None:
        path.append(parent_node)
        current_node = parent_node
        parent_node = parent_map.get(current_node)
        
    return path

def removeSeen(children, open_list, closed_list):
    """Filters out children that have already been seen."""
    seen_in_open = {node for node, parent in open_list}
    seen_in_closed = {node for node, parent in closed_list}
    
    new_nodes = [child for child in children if child not in seen_in_open and child not in seen_in_closed]
            
    return new_nodes


# --- Search Algorithm Implementations ---
def bfs(start):
    """Performs a Breadth-First Search (BFS) to find the shortest path."""
    OPEN = [(start, None)]
    CLOSED = []

    while OPEN:
        node_pair = OPEN.pop(0)  
        N, parent = node_pair
        
        if N.goalTest():
            print("Goal found using BFS!")
            path = reconstructPath(node_pair, CLOSED)
            path.reverse()
            
            print("\n--- BFS Solution Path (Shortest) ---")
            for i, p in enumerate(path):
                print(f"Step {i:>2}: Total Time: {p.time_elapsed:<2} min | {p}")
            
            return
        else:
            CLOSED.append(node_pair)
            children = N.moveGen()
            new_nodes = removeSeen(children, OPEN, CLOSED)
            new_pairs = [(child, N) for child in new_nodes]
            OPEN.extend(new_pairs) # Add to the end of the list
    
    print(" No solution found with BFS.")

def dfs(start):
    
    OPEN = [(start, None)]
    CLOSED = []

    while OPEN:
        node_pair = OPEN.pop(0) # LIFO behavior when adding to front
        N, parent = node_pair
        
        if N.goalTest():
            print(" Goal found using DFS!")
            path = reconstructPath(node_pair, CLOSED)
            path.reverse()
            
            print("\n--- DFS Solution Path ---")
            for i, p in enumerate(path):
                print(f"Step {i:>2}: Total Time: {p.time_elapsed:<2} min | {p}")
            
            return
        else:
            CLOSED.append(node_pair)
            children = N.moveGen()
            new_nodes = removeSeen(children, OPEN, CLOSED)
            new_pairs = [(child, N) for child in new_nodes]
            OPEN = new_pairs + OPEN # Add to the front of the list for stack behavior
    
    print(" No solution found with DFS.")


# --- Main execution block ---
if __name__ == "__main__":
    
    start_people = list(PEOPLE.keys())
    start_node = State(start_people, True, 0)

    # --- Run BFS ---
    print("--- 1. Solving with Breadth-First Search (BFS) ---")
    print("BFS guarantees finding the solution with the fewest trips.\n")
    bfs(start_node)
    
    print("\n")

    # --- Run DFS ---
    print("--- 2. Solving with Depth-First Search (DFS) ---")
    print("DFS explores deeply and finds a solution, but not necessarily the fastest one.\n")
    dfs(start_node)

 

--- 1. Solving with Breadth-First Search (BFS) ---
BFS guarantees finding the solution with the fewest trips.

Goal found using BFS!

--- BFS Solution Path (Shortest) ---
Step  0: Total Time: 0  min | Start Side: ['Ameya', 'Amogh', 'Grandfather', 'Grandmother'], End Side: ['(None)'], Umbrella at: START
Step  1: Total Time: 10 min | Start Side: ['Grandfather', 'Grandmother'], End Side: ['Ameya', 'Amogh'], Umbrella at: END
Step  2: Total Time: 15 min | Start Side: ['Amogh', 'Grandfather', 'Grandmother'], End Side: ['Ameya'], Umbrella at: START
Step  3: Total Time: 40 min | Start Side: ['Amogh'], End Side: ['Ameya', 'Grandfather', 'Grandmother'], Umbrella at: END
Step  4: Total Time: 50 min | Start Side: ['Ameya', 'Amogh'], End Side: ['Grandfather', 'Grandmother'], Umbrella at: START
Step  5: Total Time: 60 min | Start Side: ['(None)'], End Side: ['Ameya', 'Amogh', 'Grandfather', 'Grandmother'], Umbrella at: END


--- 2. Solving with Depth-First Search (DFS) ---
DFS explores deeply and fi